# | Extraction of Respiratory Pattern using PBM |

**References**
- [paper](https://people.csail.mit.edu/billf/publications/Phase-Based_Video.pdf)
- https://machinelearningmastery.com/time-series-data-stationary-python/

---

## Import Modules

In [ ]:
import os
import numpy as np
from tqdm import tqdm
import copy

import cv2
import imageio as iio
import mediapy as media

import matplotlib.pyplot as plt 

In [ ]:
import skimage.transform as sktransform
from skimage.filters import gaussian
import pyfftw.interfaces.scipy_fftpack as sfft

---

## [ Extract Respiratory Pattern ]

### 01. Data preparation

* loadFrames - fps 추가
* load_MultipleFrames 구현 (frame_dir를 list형태로 여러개 받기)

In [ ]:
# =========================================================================== #
def loadVideo(file_dir):
    '''
    Load RGB video using imageio (uint8)
    - file_dir: video file path (ex> end with ".mp4")
    '''
    reader = iio.get_reader(file_dir)
    orig_vid = []
    for i, im in tqdm(enumerate(reader), 
                      ascii=True, 
                      desc="Load Video"):
        orig_vid.append(im)
        
    return np.asarray(orig_vid)


def FrameResize(video, size):
    '''
    Resize video frames (ndarray)
    - video: array of video frames (num_frames, width, height, channel)
    - size: (width, height)
    '''
    w, h = size
    frames_resized = np.zeros((len(video), w, h, 3))
    for i in tqdm(range(len(video)),
                  ascii=True, 
                  desc="Frame Resizing"):
        frames_resized[i] = sktransform.resize(video[i], size)
        
    return frames_resized
# =========================================================================== #   

# =========================================================================== #
def loadFrames(frames_dir, ratio, size=None):
    '''
    Load RGB video frames (uint8)
    - frames_dir: frames directory path (frames: ".jpg")
    - size: (width, height)
    '''    
    frame_names = [f for f in sorted(os.listdir(frames_dir)) if f.endswith('.jpg')]
    
    num = int(len(frame_names)*ratio)
    interval = int(1 / ratio)
    frames = []
    
    for i in tqdm(range(num), 
                  ascii=True, 
                  desc="Load Frames"):

        frame_file = os.path.join(frames_dir, frame_names[i*interval])
        
        # Read frame
        frame = cv2.imread(frame_file)
        
        # Crop
        frame = frame[32:422, 125:515]
        #frame = frame[52:402, 145:495]
        # BGR to RGB
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        # Resize
        if size != None:
            w, h = size
            frame = cv2.resize(frame, (w, h))
        # Normalize
        frame = frame / 255.
        
        frames.append(frame)

    return np.array(frames)
# =========================================================================== #

# # Load RGB video using opencv (uint8)
# def load_video(video_path):
#     image_sequence = []
#     video = cv2.VideoCapture(video_path)
#     fps = video.get(cv2.CAP_PROP_FPS)

#     while video.isOpened():
#         ret, frame = video.read()
#         if ret is False:
#             break
#         image_sequence.append(frame[:, :, ::-1])

#     video.release()

#     return np.asarray(image_sequence), fps



# def load_MultipleFrames(frames_dirs, size=None):
#     '''
#     Load RGB video frames (uint8)
#     - frames_dirs: [frame_dir1, frame_dir2, ...]
#     - size: (width, height)
#     '''    
#     frame_names = [f for f in sorted(os.listdir(frames_dir)) if f.endswith('.jpg')]
    
#     frames = []
#     for frame_idx, frame_n in tqdm(enumerate(frame_names), 
#                                    ascii=True, 
#                                    desc="Load Frames"):
#         frame_file = os.path.join(frames_dir, frame_n)
        
#         # Read frame
#         frame = cv2.imread(frame_file)
        
# #         # Preprocessing
# #         img_yuv = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
        
# #         img_eq = img_yuv.copy()
# #         img_eq[:,:,0] = cv2.equalizeHist(img_eq[:,:,0])
# #         img_eq = cv2.cvtColor(img_eq, cv2.COLOR_YUV2BGR)
        
# #         img_clahe = img_yuv.copy()
# #         clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
# #         img_clahe[:,:,0] = clahe.apply(img_clahe[:,:,0])           
# #         frame = cv2.cvtColor(img_clahe, cv2.COLOR_YUV2BGR)

#         # Crop
#         frame = frame[32:422, 125:515]
#         # BGR to RGB
#         frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         # Resize
#         if size != None:
#             w, h = size
#             frame = cv2.resize(frame, (w, h))
#         # Normalize
#         frame = frame / 255.
        
#         frames.append(frame)

#     return np.array(frames)

In [ ]:
yiq_from_rgb = (np.array([[0.29900000,  0.58700000,  0.11400000],
                          [0.59590059, -0.27455667, -0.32134392],
                          [0.21153661, -0.52273617,  0.31119955]])).astype(np.float32)
rgb_from_yiq = np.linalg.inv(yiq_from_rgb)


def rgb2yiq(rgb_image):
    image = rgb_image.astype(np.float32)
    #return image.dot(yiq_from_rgb.T) 
    return image @ yiq_from_rgb.T

def yiq2rgb(yiq_image):
    image = yiq_image.astype(np.float32)
    #return image.dot(rgb_from_yiq.T)
    return image @ rgb_from_yiq.T

### 02. Complex steerable filters

In [ ]:
def simplify_phase(x):
    '''
    Moves x into the [-pi, pi] range.
    '''
    phase = ((x + np.pi) % (2*np.pi)) - np.pi
    return phase


def max_pyr_height(size):
    '''
    Gets the maximum possible steerable pyramid height
    - size: (width, height)
    '''
    return int(np.log2(min(size))) - 2

In [ ]:
def get_polar_grid(dims):
    center = np.ceil((np.array(dims))/2).astype(int)
    xramp, yramp = np.meshgrid(np.linspace(-1, 1, dims[1]+1)[:-1], np.linspace(-1, 1, dims[0]+1)[:-1])
    theta = np.arctan2(yramp, xramp)
    r = np.sqrt(xramp**2 + yramp**2)
    
    # eliminate the zero at the center
    r[center[0], center[1]] = min((r[center[0], center[1]-1], r[center[0]-1, center[1]]))/2
    
    return theta,r


def get_radial_mask_pair(r, rad, t_width):
    log_rad = np.log2(rad)-np.log2(r)
    hi_mask = abs(np.cos(log_rad.clip(min=-t_width, max=0)*np.pi/(2*t_width)))
    lo_mask = np.sqrt(1-(hi_mask**2))
    
    return (hi_mask, lo_mask)


def get_angle_mask(b, orientations, angle):
    order = orientations - 1
    a_constant = np.sqrt((2**(2*order))*(np.math.factorial(order)**2)/(orientations*np.math.factorial(2*order)))
    angle2 = simplify_phase(angle - (np.pi*b/orientations))
    return 2*a_constant*(np.cos(angle2)**order)*(abs(angle2) < np.pi/2)


def get_filters(dims, r_vals=None, orientations=2, only_ver=False, t_width=1):
    """
    Gets a steerbale filter bank (ndarrays)
    - dims: (h, w). Dimensions of the output filters. 
            Should be the same size as the image you're using these to filter
    - r_vals: The boundary between adjacent filters. 
              Should be an array.
              e.g.: 2**np.array(list(range(0,-7,-1)))
    - orientations: The number of filters per level
    - t-width: The falloff of each filter. 
               Smaller t_widths correspond to thicker filters with less falloff
    - only_ver: get filters of vertical direction.
    """
    if r_vals is None:
        r_vals = 2**np.array(list(range(0,-max_pyr_height(dims)-1,-1)), dtype=float)
        
    angle, r = get_polar_grid(dims)
    hi_mask, lo_mask_prev = get_radial_mask_pair(r_vals[0], r, t_width)
    
    filters = [hi_mask]
    for i in range(1, len(r_vals)):
        hi_mask, lo_mask = get_radial_mask_pair(r_vals[i], r, t_width)
        rad_mask = hi_mask * lo_mask_prev
        
        for j in range(orientations):
            angle_mask = get_angle_mask(j, orientations, angle)
            angle_mask = np.rot90(angle_mask, 2) # add for rotation of filter
            filters += [rad_mask*angle_mask/2]
        lo_mask_prev = lo_mask
        
    filters += [lo_mask_prev]
    
    if only_ver == True:
        fil_idx = [i for i in range(len(filters)) if (i % 2 == 0)]
        fil_idx.append(len(filters)-1)
        filters = [filters[idx] for idx in fil_idx]
    
    return filters

### 03. Temporal filter

In [ ]:
def difference_of_iir(delta, rl, rh):
    lowpass_1 = delta[0].copy()
    lowpass_2 = lowpass_1.copy()
    out = np.zeros(delta.shape, dtype=delta.dtype)
    for i in range(1, delta.shape[0]):
        lowpass_1 = (1-rh)*lowpass_1 + rh*delta[i]
        lowpass_2 = (1-rl)*lowpass_2 + rl*delta[i]
        out[i] = lowpass_1 - lowpass_2
    return out

In [ ]:
def amplitude_weighted_blur(x, weight, sigma):
    '''
    Where x is phase of the frame ,weight is total amplitude of the frame, 
    sigma is Standard deviation for Gaussian kernel.
    The mode parameter determines how the array borders are handled.
    '''
    if sigma != 0:
        return gaussian(x*weight, sigma, mode="wrap") / gaussian(weight, sigma, mode="wrap")
    return x

### 04. Phase based magnification

In [ ]:
def motionMag(video, mag_factor, freq_range, attenuate, sigma, temporal_filter):
    
    # Video Info
    num_frames, w, h, num_ch = video.shape
    
    # Get YIQ video
    yiq_video = rgb2yiq(video)
    print("YIQ video: Done.")
    
    # Get FFT video
    fft_video = np.zeros((num_frames, w, h), dtype=np.complex64)
    for i in tqdm(range(num_frames),
                  ascii=True,
                  desc="FFT video"):
        fft_video[i] = sfft.fftshift(sfft.fft2(yiq_video[i][:,:,0]))
    
    # Pyramid height
    pyr_height = max_pyr_height((w, h))
    print("Pyramid height: ", pyr_height)
    
    # Complex Steerable Filters
    filters = get_filters((w, h), 2**np.array(list(range(0,-pyr_height-1,-1)), dtype=float), 2, True)
    print("Number of Filters: ", len(filters))
    
    
    # Magnification
    
    ## Magnified y channel
    mag_y_channel = np.zeros((num_frames, w, h), dtype=np.complex64)
    ## y channel w/o magnification
    y_channel = np.zeros((num_frames, w, h), dtype=np.complex64)
    
    dc_frame_idx = 0
    fl, fh = freq_range
    for i in tqdm(range(1,len(filters)-1),
                  ascii=True,
                  desc="Bandpassing"):

        dc_frame = sfft.ifft2(sfft.ifftshift(filters[i]*fft_video[dc_frame_idx]))    
        dc_frame_no_mag = dc_frame / np.abs(dc_frame)    
        dc_frame_phase = np.angle(dc_frame)

        total = np.zeros(fft_video.shape, dtype=float)
        filtered = np.zeros(fft_video.shape, dtype=np.complex64)

        for n in range(num_frames):
            filtered[n] = sfft.ifft2(sfft.ifftshift(filters[i]*fft_video[n]))
            total[n] = simplify_phase(np.angle(filtered[n]) - dc_frame_phase)

        ## Temporal Filter
        total = temporal_filter(total, fl, fh).astype(float)

        for n in range(num_frames):
            phase_of_frame = total[n]
            if sigma != 0:
                phase_of_frame = amplitude_weighted_blur(phase_of_frame, np.abs(filtered[n]), sigma)

            ## Magnified phase
            mag_phase_of_frame = phase_of_frame * mag_factor
            #phase_of_frame *= mag_factor

            if attenuate:
                temp_orig = np.abs(filtered[n])*dc_frame_no_mag
            else:
                temp_orig = filtered[n]
            
            no_mag_component = 2*filters[i]*sfft.fftshift(sfft.fft2(temp_orig*np.exp(1j*phase_of_frame)))
            magnified_component = 2*filters[i]*sfft.fftshift(sfft.fft2(temp_orig*np.exp(1j*mag_phase_of_frame)))

            y_channel[n] = y_channel[n] + no_mag_component
            mag_y_channel[n] = mag_y_channel[n] + magnified_component

            
    for i in tqdm(range(num_frames),
                  ascii=True,
                  desc="Make magnified pyramids"):
        y_channel[i] = y_channel[i] + (fft_video[i]*(filters[-1]**2))
        mag_y_channel[i] = mag_y_channel[i] + (fft_video[i]*(filters[-1]**2))

            
    out_mag = np.zeros(yiq_video.shape)
    out_no_mag = np.zeros(yiq_video.shape)
    for i in tqdm(range(num_frames),
                  ascii=True,
                  desc="Video Reconstruction"):
        out_frame  = np.dstack((np.real(sfft.ifft2(sfft.ifftshift(y_channel[i]))), yiq_video[i,:,:,1:3])) 
        out_mag_frame  = np.dstack((np.real(sfft.ifft2(sfft.ifftshift(mag_y_channel[i]))), yiq_video[i,:,:,1:3]))
        
        out_no_mag[i] = out_frame
        out_mag[i] = out_mag_frame
                  
    out_no_mag = yiq2rgb(out_no_mag)
    out_mag = yiq2rgb(out_mag)
    
    return out_no_mag, out_mag
    #return out_no_mag.clip(min=0, max=1), out_mag.clip(min=0, max=1)
    #return out

### 05. Movement extraction

In [ ]:
def frame_difference(no_mag, mag):
    
    # Calculate frame difference
    diff = np.abs(mag - no_mag)
    
    # Difference sum
    diff_sum = diff.sum(axis=0)
    diff_sum = diff_sum.sum(axis=-1)
    
    return diff_sum


def vis_fdiff(orig, diff):
    
    plt.subplot(121),plt.imshow(orig, cmap = 'gray')
    plt.title('Frame'), plt.xticks([]), plt.yticks([])
    plt.subplot(122),plt.imshow(diff, cmap = 'gray')
    plt.title('Movement'), plt.xticks([]), plt.yticks([])
    plt.show()

In [ ]:
def get_max_point(diff):
    
    max_pt = np.where(diff == np.max(diff))
    
    return max_pt[0][0], max_pt[1][0]


def draw_max_pt(frame, max_point):
    
    center_coordinates = max_point[1], max_point[0]
    pt_on_frame = copy.deepcopy(frame)
    pt_on_frame = (255 * pt_on_frame).astype(np.uint8)
    pt_on_frame = cv2.circle(pt_on_frame, center_coordinates, radius=3, color=(255,0,0), thickness=-1)
    
    return pt_on_frame

In [ ]:
def get_movement_signal(video, max_point):
    
    p1, p2 = max_point
    traj = video[:, p1, p2, :]
    movement = traj.sum(axis=-1)
    
    mov_normalized = (movement - np.mean(movement)) / np.std(movement)
    
    return movement, mov_normalized


def vis_movement(movement, num_frames, fps):
    
    x = list(range(num_frames))
    time = np.array(x) / fps
    
    y_max = np.max(movement) + 0.2*np.max(movement)
    y_min = np.min(movement) - 0.2*np.abs(np.min(movement))

    plt.plot(time, movement)
    plt.xlabel('time (s)', fontsize=15)
    plt.ylim([y_min, y_max])
    plt.show()

### 07. Visualization

In [ ]:
def frames_with_max_pt(frames, max_point, save_dir):

    for i in tqdm(range(len(frames)),
                  ascii=True,
                  desc='Saving frames'):
        frame = frames[i]
        pt_on_frame = draw_max_pt(frame, max_point)

        # Convert RGB to BGR for save
        pt_on_frame = cv2.cvtColor(pt_on_frame, cv2.COLOR_RGB2BGR)
        
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, 'w_pt_frame_{}.jpg'.format(i))
        
        cv2.imwrite(save_path, pt_on_frame)

In [ ]:
def signal_with_pt(movement, num_frames, fps, save_dir):
    
    x = list(range(num_frames))
    time = np.array(x) / fps
    
    y_max = np.max(movement) + 0.2*np.max(movement)
    y_min = np.min(movement) - 0.2*np.abs(np.min(movement))
    
    os.makedirs(save_dir, exist_ok=True)

    for i in tqdm(range(len(x)),
                  ascii=True,
                  desc='Saving signals'):
        save_path = os.path.join(save_dir, 'signal_{}.jpg'.format(i))
        
        plt.figure(figsize=(10,5))
        #plt.plot(x, movement)
        plt.plot(time, movement)
        #plt.xticks(time, fontsize=15)
        #plt.yticks([])
    
        #plt.plot(x[i], movement[i], 'ro')
        plt.plot(time[i], movement[i], 'ro')
        
        
        #plt.xlabel('time', fontsize=15)
        
        #plt.ylim([y_min, y_max])
    
        plt.savefig(save_path, bbox_inches='tight')
        plt.close()

In [ ]:
def concatenate(num_frames, frame_dir, signal_dir):
    
    concat = []
    
    for i in tqdm(range(num_frames)):
        frame_path = os.path.join(frame_dir, 'w_pt_frame_{}.jpg'.format(i))
        frame_img = cv2.imread(frame_path)
        frame_img = cv2.cvtColor(frame_img, cv2.COLOR_BGR2RGB)
        frame_img = cv2.resize(frame_img, (300,300))
    
        signal_path = os.path.join(signal_dir, 'signal_{}.jpg'.format(i))
        signal_img = cv2.imread(signal_path)
        signal_img = cv2.cvtColor(signal_img, cv2.COLOR_BGR2RGB)
        signal_img = cv2.resize(signal_img, (600,300))
    
        img_concat = cv2.hconcat([frame_img, signal_img])
        concat.append(img_concat)
        
    return concat

### 08. Final Code

In [ ]:
def resp_extraction(video, fps, mag_factor, freq_range, attenuate, sigma, temporal_filter, save_dir):
    
    # Phase based magnification
    no_mag, mag = motionMag(video, mag_factor, freq_range, attenuate, sigma, temporal_filter)
    
    # get the maximum movement point
    diff = frame_difference(no_mag, mag)
    max_point = get_max_point(diff)
    
    # Post-processing of magnification
    mag = mag.clip(min=0, max=1)
    
    # get movement siganl
    mov, mov_normalized = get_movement_signal(video, max_point)
    mag_mov, mag_mov_normalized = get_movement_signal(mag, max_point)
    
    # save movement signal
    np.save(os.path.join(save_dir, 'movement.npy'), mov)
    np.save(os.path.join(save_dir, 'magnified_movement.npy'), mag_mov)
    
    # visualization
    
    ## frames
    frame_save_path = os.path.join(save_dir, 'frames')
    frames_with_max_pt(video, max_point, frame_save_path)
    
    mag_frame_save_path = os.path.join(save_dir, 'mag_frames')
    frames_with_max_pt(mag, max_point, mag_frame_save_path)
    
    ## signals
    num_frames = len(video)
    
    signal_save_path = os.path.join(save_dir, 'signals')
    signal_with_pt(mov_normalized, num_frames, fps, signal_save_path)
    
    mag_signal_save_path = os.path.join(save_dir, 'mag_signals')
    signal_with_pt(mag_mov_normalized, num_frames, fps, mag_signal_save_path)
    
    return max_point

---

## [ Test samples]

### 01. Sample Data

In [ ]:
PATH = '../data'
baby_video = os.path.join(PATH, 'yuna.mp4')

In [ ]:
baby_v = loadVideo(baby_video)
baby_v = FrameResize(baby_v, (250,250))
baby_v.shape

In [ ]:
PATH = '../data'
SLEEP_PATH = os.path.join(PATH, 'sleep_data')
e_09 = os.path.join(SLEEP_PATH, 'epoch_0009')
e_103 = os.path.join(SLEEP_PATH, 'epoch_0103')
e_104 = os.path.join(SLEEP_PATH, 'epoch_0104')
e_105 = os.path.join(SLEEP_PATH, 'epoch_0105')
e_152 = os.path.join(SLEEP_PATH, 'epoch_0152')
e_153 = os.path.join(SLEEP_PATH, 'epoch_0153')

In [ ]:
ep_09 = loadFrames(e_09, 1, (250, 250))
ep_103 = loadFrames(e_103, 1, (250, 250))
ep_104 = loadFrames(e_104, 1, (250, 250))
ep_105 = loadFrames(e_105, 1, (250, 250))
ep_152 = loadFrames(e_152, 1, (250, 250))
ep_153 = loadFrames(e_153, 1, (250, 250))

### 02. Case study

#### Baby

In [ ]:
video=baby_v
save_dir='./res/baby'

fps=30
mag_factor = 50
freq_range = [.2, .3]
attenuate=True # attenuate_other_frequencies
sigma = 5
temporal_filter = difference_of_iir
num_frames=301
frame_dir=os.path.join(save_dir, 'frames')
signal_dir=os.path.join(save_dir, 'signals')
mag_frame_dir=os.path.join(save_dir, 'mag_frames')
mag_signal_dir=os.path.join(save_dir, 'mag_signals')

In [ ]:
m_baby = resp_extraction(video, fps, mag_factor, freq_range, attenuate, sigma, temporal_filter, save_dir)
m_baby

In [ ]:
res_baby = concatenate(num_frames, mag_frame_dir, signal_dir)
media.show_video(res_baby, codec='gif', fps=30)

#### Epoch 09

In [ ]:
video=ep_09
save_dir='./res/epoch0009'

fps=5
mag_factor = 50
freq_range = [.2, .3]
attenuate=True # attenuate_other_frequencies
sigma = 5
temporal_filter = difference_of_iir
num_frames=150
frame_dir=os.path.join(save_dir, 'frames')
signal_dir=os.path.join(save_dir, 'signals')
mag_frame_dir=os.path.join(save_dir, 'mag_frames')
mag_signal_dir=os.path.join(save_dir, 'mag_signals')

In [ ]:
m_09 = resp_extraction(video, fps, mag_factor, freq_range, attenuate, sigma, temporal_filter, save_dir)
print(m_09)

In [ ]:
res_09 = concatenate(num_frames, mag_frame_dir, signal_dir)
media.show_video(res_09, codec='gif', fps=5)

#### Epoch 103

In [ ]:
video=ep_103
save_dir='./res/epoch0103'

fps=5
mag_factor = 50
freq_range = [.2, .3]
attenuate=True # attenuate_other_frequencies
sigma = 5
temporal_filter = difference_of_iir
num_frames=150
frame_dir=os.path.join(save_dir, 'frames')
signal_dir=os.path.join(save_dir, 'signals')
mag_frame_dir=os.path.join(save_dir, 'mag_frames')
mag_signal_dir=os.path.join(save_dir, 'mag_signals')

In [ ]:
m_103 = resp_extraction(video, fps, mag_factor, freq_range, attenuate, sigma, temporal_filter, save_dir)
print(m_103)

In [ ]:
res_103 = concatenate(num_frames, mag_frame_dir, signal_dir)
media.show_video(res_103, codec='gif', fps=5)

#### Epoch 104

In [ ]:
video=ep_104
save_dir='./res/epoch0104'

fps=5
mag_factor = 50
freq_range = [.2, .3]
attenuate=True # attenuate_other_frequencies
sigma = 5
temporal_filter = difference_of_iir
num_frames=150
frame_dir=os.path.join(save_dir, 'frames')
signal_dir=os.path.join(save_dir, 'signals')
mag_frame_dir=os.path.join(save_dir, 'mag_frames')
mag_signal_dir=os.path.join(save_dir, 'mag_signals')

In [ ]:
m_104 = resp_extraction(video, fps, mag_factor, freq_range, attenuate, sigma, temporal_filter, save_dir)
print(m_104)

In [ ]:
res_104 = concatenate(num_frames, mag_frame_dir, signal_dir)
media.show_video(res_104, codec='gif', fps=5)

#### Epoch 105

In [ ]:
video=ep_105
save_dir='./res/epoch0105'

fps=5
mag_factor = 50
freq_range = [.2, .3]
attenuate=True # attenuate_other_frequencies
sigma = 5
temporal_filter = difference_of_iir
num_frames=150
frame_dir=os.path.join(save_dir, 'frames')
signal_dir=os.path.join(save_dir, 'signals')
mag_frame_dir=os.path.join(save_dir, 'mag_frames')
mag_signal_dir=os.path.join(save_dir, 'mag_signals')

In [ ]:
m_105 = resp_extraction(video, fps, mag_factor, freq_range, attenuate, sigma, temporal_filter, save_dir)
print(m_105)

In [ ]:
res_105 = concatenate(num_frames, mag_frame_dir, signal_dir)
media.show_video(res_105, codec='gif', fps=5)

#### Epoch 152

In [ ]:
video=ep_152
save_dir='./res/epoch0152'

fps=5
mag_factor = 50
freq_range = [.2, .3]
attenuate=True # attenuate_other_frequencies
sigma = 5
temporal_filter = difference_of_iir
num_frames=150
frame_dir=os.path.join(save_dir, 'frames')
signal_dir=os.path.join(save_dir, 'signals')
mag_frame_dir=os.path.join(save_dir, 'mag_frames')
mag_signal_dir=os.path.join(save_dir, 'mag_signals')

In [ ]:
m_152 = resp_extraction(video, fps, mag_factor, freq_range, attenuate, sigma, temporal_filter, save_dir)
print(m_152)

In [ ]:
res_152 = concatenate(num_frames, mag_frame_dir, signal_dir)
media.show_video(res_152, codec='gif', fps=5)

#### Epoch 153

In [ ]:
video=ep_153
save_dir='./res/epoch0153'

fps=5
mag_factor = 50
freq_range = [.2, .3]
attenuate=True # attenuate_other_frequencies
sigma = 5
temporal_filter = difference_of_iir
num_frames=150
frame_dir=os.path.join(save_dir, 'frames')
signal_dir=os.path.join(save_dir, 'signals')
mag_frame_dir=os.path.join(save_dir, 'mag_frames')
mag_signal_dir=os.path.join(save_dir, 'mag_signals')

In [ ]:
m_153 = resp_extraction(video, fps, mag_factor, freq_range, attenuate, sigma, temporal_filter, save_dir)
print(m_153)

In [ ]:
res_153 = concatenate(num_frames, mag_frame_dir, signal_dir)
media.show_video(res_153, codec='gif', fps=5)

---

## [ Object tracking ]

In [ ]:
def tracking_movement(video, max_point, bbx_size):
    
    p2, p1 = max_point
    roi = video.copy()
    
    bbx_size = bbx_size // 2
    print(bbx_size)
    hl, hh, wl, wh = p1-bbx_size, p1+bbx_size, p2-bbx_size, p2+bbx_size
    print(hl, hh, wl, wh)
    
    roi = (roi[:, wl:wh, hl:hh]*255).astype(np.uint8)
    print(roi.shape)
    new_pt = np.array([[[bbx_size, bbx_size]]]).astype(np.float32)
    
    color=np.array([0,255,0])

    
    # Take first frame 
    first_frame = roi[0]
    prev_gray = cv2.cvtColor(first_frame, cv2.COLOR_RGB2GRAY)

    prev = new_pt

    # Create a mask image for drawing purposes
    mask = np.zeros_like(first_frame)

    #x_val = [prev[0][0][0]]
    y_val = [prev[0][0][1]]
    for j in tqdm(range(1, len(roi)),
                  ascii=True,
                  desc="Estimate movement"):
        # calculate optical flow
        frame = roi[j]
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        #nxt, status, error = cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev, None, **lk_params)
        nxt, status, error = cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev, None)

        good_new = nxt[0]
        good_old = prev[0]
    
        # draw the tracks
        for i,(new,old) in enumerate(zip(good_new,good_old)):
            a,b = new.ravel()
            c,d = old.ravel()
            mask = cv2.line(mask, (int(a),int(b)),(int(c),int(d)), color.tolist(), 2)
            frame = cv2.circle(frame,(int(a),int(b)),2,color.tolist(),-1)
            #x_val.append(a)
            y_val.append(b)
            
        img = cv2.add(frame, mask)

        # concatenate frames 
        if j == 1:
            of_video = img[np.newaxis,...]
        else:
            of_video = np.concatenate((of_video, img[np.newaxis]))
        
        # Now update the previous frame and previous points
        prev_gray = gray.copy()
        prev = good_new.reshape(-1,1,2)
        
    return of_video, y_val


# def tracking_movement(video, max_point, bbx_size):
    
#     p2, p1 = max_point
#     roi = video.copy()
    
#     bbx_size = bbx_size // 2
#     hl, hh, wl, wh = p1-bbx_size, p1+bbx_size, p2-bbx_size, p2+bbx_size
    
#     roi = (roi[:, wl:wh, hl:hh]*255).astype(np.uint8)
#     new_pt = np.array([[[bbx_size, bbx_size]]]).astype(np.float32)
    
#     color=np.array([0,255,0])

#     # Take first frame 
#     first_frame = roi[0]
#     prev_gray = cv2.cvtColor(first_frame, cv2.COLOR_RGB2GRAY)

#     prev = new_pt

#     # Create a mask image for drawing purposes
#     mask = np.zeros_like(first_frame)

#     #x_val = [prev[0][0][0]]
    
#     y_val = [prev[0][0][1]]
#     for j in tqdm(range(1, len(roi))):
#         # calculate optical flow
#         frame = roi[j]
#         gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
#         #nxt, status, error = cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev, None, **lk_params)
#         nxt, status, error = cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev, None)

#         good_new = nxt[0]
#         good_old = prev[0]
    
#         # draw the tracks
#         for i,(new,old) in enumerate(zip(good_new,good_old)):
#             a,b = new.ravel()
#             c,d = old.ravel()
#             mask = cv2.line(mask, (int(a),int(b)),(int(c),int(d)), color.tolist(), 2)
#             frame = cv2.circle(frame,(int(a),int(b)),2,color.tolist(),-1)
            
#             sign = np.sign(b - d)
#             dist = np.sqrt(np.square(a-c)+np.square(b-d))
#             #x_val.append(a)
#             y_val.append(y_val[-1] + (sign*dist))
            
#         img = cv2.add(frame, mask)

#         # concatenate frames 
#         if j == 1:
#             of_video = img[np.newaxis,...]
#         else:
#             of_video = np.concatenate((of_video, img[np.newaxis]))
        
#         # Now update the previous frame and previous points
#         prev_gray = gray.copy()
#         prev = good_new.reshape(-1,1,2)
        
#     return of_video, y_val

In [ ]:
of_video, y_val = tracking_movement(baby_v, m_baby, bbx_size=50)

In [ ]:
media.show_videos(of_video[np.newaxis,0:300,...], codec='gif', fps=10)

In [ ]:
plt.plot(list(range(len(y_val))), y_val)

---

## Post-processing

In [ ]:
from scipy.signal import butter, sosfilt, iirnotch, lfilter
from sklearn.preprocessing import scale

def butter_bandpass(lowcut, highcut, fs, order=1):
    low = lowcut
    high = highcut
    sos  = butter(order, [low, high], btype='band', fs=fs, output='sos', analog=False)
    return sos 

def butter_bandpass_filter(data, lowcut, highcut, fs, order=1):
    sos  = butter_bandpass(lowcut, highcut, fs, order=order)
    y = sosfilt(sos, data)
    return y

def notch_filter(data, f0, Q, fs):
    b, a = iirnotch(f0, Q, fs)
    y = lfilter(b, a, data)
    return y

def filtered_signal(signal, sample_freq, low=0.1, high=0.5, order=1):
    
#     # Notch filter to cancel out the power line disturbance (50 Hz - 60Hz)
#     f0, Q = 50, 5
#     y1 = notch_filter(signal, f0, Q, fs=sample_freq)
#     f0 = 60
#     y2 = notch_filter(y1, f0, Q, fs=sample_freq)
    
    #y2 = signal
    # Butterworth filter for valid freq signals
    #filtered_signal = butter_bandpass_filter(y2, low, high, fs=sample_freq, order=order)
    filtered_signal = butter_bandpass_filter(signal, low, high, fs=sample_freq, order=order)
    
    return filtered_signal

def normalize(y_val):
    y = (y_val - np.min(y_val)) / (np.max(y_val) - np.min(y_val))
    return y


def processing(signal):
    standardize = scale(signal)
    butter = filtered_signal(standardize, 5)
    norm = normalize(butter)
    #norm = normalize(standardize)
    
    return norm

In [ ]:
of_video, y_val = tracking_movement(ep_09, m_09, bbx_size=50)

In [ ]:
media.show_videos(of_video[np.newaxis,...], codec='gif', fps=10)

In [ ]:
plt.plot(list(range(len(y_val))), y_val)

In [ ]:
y = processing(y_val)

In [ ]:
y = processing(y_val)
plt.plot(list(range(len(y))), y)

In [ ]:
of_video, y_val = tracking_movement(ep_152, m_152, bbx_size=50)

In [ ]:
media.show_videos(of_video[np.newaxis,...], codec='gif', fps=10)

In [ ]:
plt.plot(list(range(len(y_val))), y_val)

In [ ]:
y = processing(y_val)

In [ ]:
plt.plot(list(range(len(y))), y)

In [ ]:
of_video, y_val = tracking_movement(ep_103, m_103, bbx_size=50)

In [ ]:
media.show_videos(of_video[np.newaxis,...], codec='gif', fps=10)

In [ ]:
plt.plot(list(range(len(y_val))), y_val)

In [ ]:
y = processing(y_val)
plt.plot(list(range(len(y))), y)

---

## [ Case Study! ] 

### 01. Load frames (2.5 fps) & Ground truth data

**Videos**

In [ ]:
ep_9 = loadFrames(e_09, 0.5, (250, 250))
ep_103 = loadFrames(e_103, 0.5, (250, 250))
ep_104 = loadFrames(e_104, 0.5, (250, 250))
ep_105 = loadFrames(e_105, 0.5, (250, 250))
ep_152 = loadFrames(e_152, 0.5, (250, 250))
ep_153 = loadFrames(e_153, 0.5, (250, 250))

**Ground Truth**

In [ ]:
GT_PATH = './gt'
os.path.exists(GT_PATH)

-

### 02. Processed Data

In [ ]:
def result(video, max_point, gt, fps):
    
    res_v, y_val = tracking_movement(video, max_point, bbx_size=50)
    
    time = np.array(list(range(len(y_val)))) / fps
    y = processing(y_val)
    
    y_gt_raw = np.load(gt)
    
    n_frames = len(y) 
    intv = len(y_gt_raw)//n_frames
    
    y_gt = []
    for i in range(n_frames):
        y_gt.append(y_gt_raw[i*intv])
    y_gt = np.array(y_gt)
    
    time_gt = np.array(list(range(len(y_gt)))) / fps
    
    plt.plot(time_gt, y_gt, label='ground truth')
    plt.plot(time, y, label='estimated')
    plt.legend()
    plt.show()
    
    return time, y, time_gt, y_gt

In [ ]:
fps = 2.5

In [ ]:
epoch = 9
video = ep_9
max_point = m_09
gt = os.path.join(GT_PATH, str(epoch)+'.npy')
time9, y9, time_gt9, y_gt9 = result(video, max_point, gt, fps)

In [ ]:
epoch = 103
video = ep_103
max_point = m_103
gt = os.path.join(GT_PATH, str(epoch)+'.npy')
time103, y103, time_gt103, y_gt103 = result(video, max_point, gt, fps)

In [ ]:
epoch = 104
video = ep_104
max_point = m_104
gt = os.path.join(GT_PATH, str(epoch)+'.npy')
time104, y104, time_gt104, y_gt104 = result(video, max_point, gt, fps)

In [ ]:
epoch = 105
video = ep_105
max_point = m_105
gt = os.path.join(GT_PATH, str(epoch)+'.npy')
time105, y105, time_gt105, y_gt105 = result(video, max_point, gt, fps)

In [ ]:
epoch = 152
video = ep_152
max_point = m_152
gt = os.path.join(GT_PATH, str(epoch)+'.npy')
time152, y152, time_gt152, y_gt152 = result(video, max_point, gt, fps)

In [ ]:
epoch = 153
video = ep_153
max_point = m_153
gt = os.path.join(GT_PATH, str(epoch)+'.npy')
time153, y153, time_gt153, y_gt153 = result(video, max_point, gt, fps)

-

### 03. Frequency and Amplitude

In [ ]:
from scipy.signal import find_peaks
from statsmodels.tsa.stattools import adfuller

In [ ]:
def PositiveFFT(x, Fs):
    Ts = 1/Fs
    Nsamp = x.size
    xFreq = np.fft.rfftfreq(Nsamp, Ts)[:-1]
    yFFT = (np.fft.rfft(x)/Nsamp)[:-1]*2
    return xFreq, yFFT

In [ ]:
def find_freq(y, y_gt, fps):
    
    xFreq, FFT_data = PositiveFFT(y, fps)
    peaks, prop = find_peaks(abs(FFT_data), height=0.1)
    
    xFreq_gt, FFT_data_gt = PositiveFFT(y_gt, fps)
    peaks_gt, prop_gt = find_peaks(abs(FFT_data_gt), height=0.1)
    
    freq, freq_gt = 0, 0
    if len(peaks) == 0 or len(peaks_gt) == 0:
        print("Can't find frequency")
    else:
        f_idx = np.where(prop['peak_heights'] == max(prop['peak_heights']))[0]
        freq = xFreq[peaks[f_idx[0]]]
        
        f_idx_gt = np.where(prop_gt['peak_heights'] == max(prop_gt['peak_heights']))[0]
        freq_gt = xFreq_gt[peaks_gt[f_idx_gt[0]]]

    return freq, freq_gt

In [ ]:
f, f_gt = find_freq(y9, y_gt9, fps)

In [ ]:
f

In [ ]:
f_gt

In [ ]:
from peakdetect import peakdetect

In [ ]:
peaks = peakdetect(y9, lookahead=5)

a = np.array(peaks[0])
peak_x = a[:,0].astype(int)
peak_y = a[:,1]
plt.plot(y9)
plt.plot(peak_x, peak_y, "x")

In [ ]:
peaks

In [ ]:
def find_peak(signal, fps):
    
    peaks = peakdetect(signal, lookahead=3)
    
    top = np.array(peaks[0])
    top_x = top[:,0] / fps
    top_y = top[:,1]
    
    bottom = np.array(peaks[1])
    bottom_x = bottom[:,0] / fps
    bottom_y = bottom[:,1]
    
    time = np.array(list(range(len(signal)))) / fps
    
    plt.plot(time, signal)
    plt.plot(top_x, top_y, "x")
    plt.plot(bottom_x, bottom_y, "x")
    plt.show()
    
    return (top_x, top_y), (bottom_x, bottom_y)

In [ ]:
top9, bottom9 = find_peak(y9, fps)

In [ ]:
top_gt9, bottom_gt9 = find_peak(y_gt9, fps)

In [ ]:
top9

In [ ]:
np.where(top9[0]<15)[0]

In [ ]:
def valid_freq(signal, peaks_x, fps):
    
    s_idx = len(signal) // 2
    s_1 = signal[:s_idx]
    s_2 = signal[s_idx:]
    
    p_idx = len(np.where(peaks_x < 15)[0])
    p_1 = peaks_x[:p_idx]
    p_2 = peaks_x[p_idx:]
    
    res = adfuller(signal)
    print("p_val: ", res[1])
    
    intv = []
    for i in range(len(peaks_x)-1):
        intv.append(peaks_x[i+1] - peaks_x[i])

    intv = np.array(intv)
    #intv1 = intv1 / fps
    print("intv: ", intv)
    average = np.average(intv)
    freq = 60 / average / 60
    print("freq: ", freq)
    
    print()
    res_1 = adfuller(s_1)
    p_val1 = res_1[1]
    print("p_val1: ", p_val1)

    intv1 = []
    for i in range(len(p_1)-1):
        intv1.append(p_1[i+1] - p_1[i])

    intv1 = np.array(intv1)
    #intv1 = intv1 / fps
    print("intv1: ", intv1)
    average1 = np.average(intv1)
    freq1 = 60 / average1 / 60
    print("freq1: ", freq1)
    
    print()
    res_2 = adfuller(s_2)
    p_val2 = res_2[1]
    print("p_val2: ", p_val2)

    intv2 = []
    for i in range(len(p_2)-1):
        intv2.append(p_2[i+1] - p_2[i])    
    
    intv2 = np.array(intv2)
    #intv2 = intv2 / fps
    print("intv2: ", intv2)
    average2 = np.average(intv2)
    freq2 = 60 / average2 / 60
    print("freq2: ", freq2)
    
    return (p_val1, freq1), (p_val2, freq2)

In [ ]:
res1, res2 = valid_freq(y9, top9[0], fps)

In [ ]:
res1, res2 = valid_freq(y9, bottom9[0], fps)

In [ ]:
res_gt1, res_gt2 = valid_freq(y_gt9, top_gt9[0], fps)

In [ ]:
res_gt1, res_gt2 = valid_freq(y_gt9, bottom_gt9[0], fps)

#### Epoch 9

In [ ]:
top9, bottom9 = find_peak(y9, fps)
top_gt9, bottom_gt9 = find_peak(y_gt9, fps)

In [ ]:
res1, res2 = valid_freq(y9, bottom9[0], fps)
print("="*30)
res_gt1, res_gt2 = valid_freq(y_gt9, bottom_gt9[0], fps)

#### Epoch 103

In [ ]:
top103, bottom103 = find_peak(y103, fps)
top_gt103, bottom_gt103 = find_peak(y_gt103, fps)

In [ ]:
res1, res2 = valid_freq(y103, bottom103[0], fps)
print("="*30)
res_gt1, res_gt2 = valid_freq(y_gt103, bottom_gt103[0], fps)

#### Epoch 104

In [ ]:
top104, bottom104 = find_peak(y104, fps)
top_gt104, bottom_gt104 = find_peak(y_gt104, fps)

In [ ]:
res1, res2 = valid_freq(y104, bottom104[0], fps)
print("="*30)
res_gt1, res_gt2 = valid_freq(y_gt104, bottom_gt104[0], fps)

#### Epoch 105

In [ ]:
top105, bottom105 = find_peak(y105, fps)
top_gt105, bottom_gt105 = find_peak(y_gt105, fps)

In [ ]:
res1, res2 = valid_freq(y105, bottom105[0], fps)
print("="*30)
res_gt1, res_gt2 = valid_freq(y_gt105, bottom_gt105[0], fps)

#### Epoch 152

In [ ]:
top152, bottom152 = find_peak(y152, fps)
top_gt152, bottom_gt152 = find_peak(y_gt152, fps)

In [ ]:
res1, res2 = valid_freq(y152, bottom152[0], fps)
print("="*30)
res_gt1, res_gt2 = valid_freq(y_gt152, bottom_gt152[0], fps)

#### 153

In [ ]:
top153, bottom153 = find_peak(y153, fps)
top_gt153, bottom_gt153 = find_peak(y_gt153, fps)

In [ ]:
res1, res2 = valid_freq(y153, bottom153[0], fps)
print("="*30)
res_gt1, res_gt2 = valid_freq(y_gt153, bottom_gt153[0], fps)

In [ ]:
int(149 / 2) * 2

---

## [Final]

In [ ]:
ep_09 = loadFrames(e_09, 1, (250, 250))
ep_103 = loadFrames(e_103, 1, (250, 250))
ep_104 = loadFrames(e_104, 1, (250, 250))
ep_105 = loadFrames(e_105, 1, (250, 250))
ep_152 = loadFrames(e_152, 1, (250, 250))
ep_153 = loadFrames(e_153, 1, (250, 250))

In [ ]:
def make_half(video):
    
    v_len = int(len(video)/2)
    v_new = []
    for i in range(v_len):
        v_new.append(video[i*2])
    v_new = np.array(v_new)
    
    return v_new

In [ ]:
ep_09_new = make_half(ep_09)

In [ ]:
ep_09_new.shape

In [ ]:
media.show_video(ep_09_new, codec='gif', fps=5)

In [ ]:
def resp_extraction_r(video, fps, mag_factor, freq_range, attenuate, sigma, temporal_filter, save_dir):
    
    os.makedirs(save_dir, exist_ok=True)
    
    # Phase based magnification
    no_mag, mag = motionMag(video, mag_factor, freq_range, attenuate, sigma, temporal_filter)
    
    # get the maximum movement point
    diff = frame_difference(no_mag, mag)
    max_point = get_max_point(diff)
    
    # Post-processing of magnification
    mag = mag.clip(min=0, max=1)
    
    # get movement siganl
    ## video
    v_new = make_half(video)
    of_video, y_val = tracking_movement(v_new, max_point, bbx_size=50)
    mov = processing(y_val)
    
#     mov, mov_normalized = get_movement_signal(video, max_point)
#     mag_mov, mag_mov_normalized = get_movement_signal(mag, max_point)
    
    # save movement signal
    np.save(os.path.join(save_dir, 'movement.npy'), mov)
    #np.save(os.path.join(save_dir, 'magnified_movement.npy'), mag_mov)
    
    # visualization
    
    ## frames
    frame_save_path = os.path.join(save_dir, 'frames')
    frames_with_max_pt(video, max_point, frame_save_path)
    
    mag_frame_save_path = os.path.join(save_dir, 'mag_frames')
    frames_with_max_pt(mag, max_point, mag_frame_save_path)
    
    ## signals
    num_frames = len(v_new)
    
    signal_save_path = os.path.join(save_dir, 'signals')
    signal_with_pt(mov, num_frames, fps, signal_save_path)
    
    return max_point

In [ ]:
video=ep_09
save_dir='./res/epoch09_f'

fps=2.5
mag_factor = 50
freq_range = [.2, .3]
attenuate=True # attenuate_other_frequencies
sigma = 5
temporal_filter = difference_of_iir
num_frames=75
frame_dir=os.path.join(save_dir, 'frames')
signal_dir=os.path.join(save_dir, 'signals')
mag_frame_dir=os.path.join(save_dir, 'mag_frames')

In [ ]:
m_9 = resp_extraction_r(video, fps, mag_factor, freq_range, attenuate, sigma, temporal_filter, save_dir)
print(m_9)

In [ ]:
res_9 = concatenate(num_frames, mag_frame_dir, signal_dir)
media.show_video(res_9, codec='gif', fps=2.5)

In [ ]:
video=ep_103
save_dir='./res/epoch103_f'

fps=2.5
mag_factor = 50
freq_range = [.2, .3]
attenuate=True # attenuate_other_frequencies
sigma = 5
temporal_filter = difference_of_iir
num_frames=75
frame_dir=os.path.join(save_dir, 'frames')
signal_dir=os.path.join(save_dir, 'signals')
mag_frame_dir=os.path.join(save_dir, 'mag_frames')

In [ ]:
m_103 = resp_extraction_r(video, fps, mag_factor, freq_range, attenuate, sigma, temporal_filter, save_dir)
print(m_103)

In [ ]:
res_103 = concatenate(num_frames, mag_frame_dir, signal_dir)
media.show_video(res_103, codec='gif', fps=2.5)

In [ ]:
video=ep_104
save_dir='./res/epoch104_f'

fps=2.5
mag_factor = 50
freq_range = [.2, .3]
attenuate=True # attenuate_other_frequencies
sigma = 5
temporal_filter = difference_of_iir
num_frames=75
frame_dir=os.path.join(save_dir, 'frames')
signal_dir=os.path.join(save_dir, 'signals')
mag_frame_dir=os.path.join(save_dir, 'mag_frames')

In [ ]:
m_104 = resp_extraction_r(video, fps, mag_factor, freq_range, attenuate, sigma, temporal_filter, save_dir)
print(m_104)

In [ ]:
res_104 = concatenate(num_frames, mag_frame_dir, signal_dir)
media.show_video(res_104, codec='gif', fps=2.5)

In [ ]:
video=ep_105
save_dir='./res/epoch105_f'

fps=2.5
mag_factor = 50
freq_range = [.2, .3]
attenuate=True # attenuate_other_frequencies
sigma = 5
temporal_filter = difference_of_iir
num_frames=75
frame_dir=os.path.join(save_dir, 'frames')
signal_dir=os.path.join(save_dir, 'signals')
mag_frame_dir=os.path.join(save_dir, 'mag_frames')

In [ ]:
m_105 = resp_extraction_r(video, fps, mag_factor, freq_range, attenuate, sigma, temporal_filter, save_dir)
print(m_105)

In [ ]:
res_105 = concatenate(num_frames, mag_frame_dir, signal_dir)
media.show_video(res_105, codec='gif', fps=2.5)

In [ ]:
video=ep_152
save_dir='./res/epoch152_f'

fps=2.5
mag_factor = 50
freq_range = [.2, .3]
attenuate=True # attenuate_other_frequencies
sigma = 5
temporal_filter = difference_of_iir
num_frames=75
frame_dir=os.path.join(save_dir, 'frames')
signal_dir=os.path.join(save_dir, 'signals')
mag_frame_dir=os.path.join(save_dir, 'mag_frames')

In [ ]:
m_152 = resp_extraction_r(video, fps, mag_factor, freq_range, attenuate, sigma, temporal_filter, save_dir)
print(m_152)

In [ ]:
res_152 = concatenate(num_frames, mag_frame_dir, signal_dir)
media.show_video(res_152, codec='gif', fps=2.5)

In [ ]:
video=ep_153
save_dir='./res/epoch0153_f'

fps=2.5
mag_factor = 50
freq_range = [.2, .3]
attenuate=True # attenuate_other_frequencies
sigma = 5
temporal_filter = difference_of_iir
num_frames=75
frame_dir=os.path.join(save_dir, 'frames')
signal_dir=os.path.join(save_dir, 'signals')
mag_frame_dir=os.path.join(save_dir, 'mag_frames')

In [ ]:
m_153 = resp_extraction_r(video, fps, mag_factor, freq_range, attenuate, sigma, temporal_filter, save_dir)
print(m_153)

In [ ]:
res_153 = concatenate(num_frames, mag_frame_dir, signal_dir)
media.show_video(res_153, codec='gif', fps=2.5)

---

In [ ]:
peaks, prop = find_peaks(y9, height=0.1)
intv = []
for i in range(len(peaks)-1):
    intv.append(peaks[i+1] - peaks[i])

intv = np.array(intv)
intv = intv / 2.5
print(intv)
average = np.average(intv)
freq = 60 / average / 60
print(freq)

plt.plot(y9)
plt.plot(peaks, y9[peaks], "x")

In [ ]:
peaks, prop = find_peaks(y_gt9, height=0.1)
intv = []
for i in range(len(peaks)-1):
    intv.append(peaks[i+1] - peaks[i])

intv = np.array(intv)
intv = intv / 2.5
print(intv)
average = np.average(intv)
freq = 60 / average / 60
print(freq)

plt.plot(y_gt9)
plt.plot(peaks, y_gt9[peaks], "x")

In [ ]:
result_gt = adfuller(y9)
print('ADF Statistic: %f' % result_gt[0])
print('p-value: %f' % result_gt[1])
print('Critical Values:')
for key, value in result_gt[4].items():
    print('\t%s: %.3f' % (key, value))

In [ ]:
result_gt = adfuller(y9[:37])
print('ADF Statistic: %f' % result_gt[0])
print('p-value: %f' % result_gt[1])
print('Critical Values:')
for key, value in result_gt[4].items():
    print('\t%s: %.3f' % (key, value))

In [ ]:
result_gt = adfuller(y_gt9[37:])
print('ADF Statistic: %f' % result_gt[0])
print('p-value: %f' % result_gt[1])
print('Critical Values:')
for key, value in result_gt[4].items():
    print('\t%s: %.3f' % (key, value))

In [ ]:
from scipy.fftpack import fftfreq

In [ ]:
frequencies = fftfreq

- https://lifelong-education-dr-kim.tistory.com/entry/%ED%8C%8C%EC%9D%B4%EC%8D%AC%EC%9D%84-%EC%9D%B4%EC%9A%A9%ED%95%9C-%EC%A7%84%EB%8F%99-%EB%8D%B0%EC%9D%B4%ED%84%B0%EC%9D%98-%EA%B3%A0%EC%9C%A0-%EC%A7%84%EB%8F%99%EC%88%98-%EA%B3%84%EC%82%B0%ED%95%98%EA%B8%B0

- https://blog.ytotech.com/2015/11/01/findpeaks-in-python/

In [ ]:
plt.plot(y9)

In [ ]:
from peakdetect import peakdetect
peaks = peakdetect(y9, lookahead=5)

a = np.array(peaks[0])
peak_x = a[:,0].astype(int)
peak_y = a[:,1]
plt.plot(y9)
plt.plot(peak_x, peak_y, "x")

In [ ]:
x = a[:,0].astype(int)

In [ ]:
x

In [ ]:
a[:,1]

---

In [ ]:
of_video, y_val = tracking_movement(ep_103, m_103, bbx_size=50)
time = np.array(list(range(len(y)))) / 2.5
y = processing(y_val)
plt.plot(time, y)

In [ ]:
epoch = 103
load = os.path.join(path, str(epoch)+'.npy')
os.path.exists(load)

y_gt = np.load(load)
time_gt = np.array(list(range(len(y_gt)))) / 200
time_gt

plt.plot(time, y, time_gt, y_gt)

In [ ]:
of_video, y_val = tracking_movement(ep_104, m_104, bbx_size=50)
time = np.array(list(range(len(y)))) / 2.5
y = processing(y_val)
plt.plot(time, y)

In [ ]:
epoch = 104
load = os.path.join(path, str(epoch)+'.npy')
os.path.exists(load)

y_gt = np.load(load)
time_gt = np.array(list(range(len(y_gt)))) / 200
time_gt

plt.plot(time, y, time_gt, y_gt)

In [ ]:
of_video, y_val = tracking_movement(ep_105, m_105, bbx_size=50)
time = np.array(list(range(len(y)))) / 2.5
y = processing(y_val)
plt.plot(time, y)

In [ ]:
epoch = 105
load = os.path.join(GT_PATH, str(epoch)+'.npy')
os.path.exists(load)

y_gt = np.load(load)
time_gt = np.array(list(range(len(y_gt)))) / 200
time_gt

plt.plot(time, y, time_gt, y_gt)

In [ ]:
y_new = []
for i in range(75):
    y_new.append(y_gt[i*80])
y_new = np.array(y_new)
time_new = np.array(list(range(len(y_new)))) / 2.5

In [ ]:
plt.plot(time, y, time_new, y_new)

In [ ]:
result = adfuller(y)
print('ADF Statistic: %f' % result[0])
print('p-value: %f' % result[1])
print('Critical Values:')
for key, value in result[4].items():
    print('\t%s: %.3f' % (key, value))

In [ ]:
result_gt = adfuller(y_new[:(75//2)])
print('ADF Statistic: %f' % result_gt[0])
print('p-value: %f' % result_gt[1])
print('Critical Values:')
for key, value in result_gt[4].items():
    print('\t%s: %.3f' % (key, value))

In [ ]:
len(y_new)

---------

In [ ]:
GT_PATH = './gt'

In [ ]:
of_video, y_val = tracking_movement(ep_152, m_152, bbx_size=50)
time = np.array(list(range(len(y)))) / 2.5
y = processing(y_val)
plt.plot(time, y)

In [ ]:
epoch = 152
load = os.path.join(GT_PATH, str(epoch)+'.npy')
os.path.exists(load)

y_gt = np.load(load)
time_gt = np.array(list(range(len(y_gt)))) / 200
time_gt

plt.plot(time, y, time_gt, y_gt)

In [ ]:
y_new = []
for i in range(75):
    y_new.append(y_gt[i*80])

In [ ]:
y_new = []
for i in range(75):
    y_new.append(y_gt[i*80])
y_new = np.array(y_new)
time_new = np.array(list(range(len(y_new)))) / 2.5

In [ ]:
plt.plot(time_new, y_new, time, y)
#plt.plot(time_new, y_new)

In [ ]:
def PositiveFFT(x, Fs):
    Ts = 1/Fs
    Nsamp = x.size
    xFreq = np.fft.rfftfreq(Nsamp, Ts)[:-1]
    yFFT = (np.fft.rfft(x)/Nsamp)[:-1]*2
    return xFreq, yFFT

In [ ]:
xFreq1, FFT_data1 = PositiveFFT(y, 2.5)

In [ ]:
len(FFT_data1)

In [ ]:
plt.plot(xFreq1, abs(FFT_data1))

In [ ]:
plt.plot(abs(FFT_data1))

In [ ]:
xFreq1, FFT_data1 = PositiveFFT(y, 2.5)
peaks, prop = find_peaks(abs(FFT_data1), height=0.1)

xFreq1[7]
plt.plot(y_gt)
plt.plot(peaks, y_gt[peaks], "x")

In [ ]:
prop["peak_heights"]

In [ ]:
peaks

In [ ]:
xFreq1[7]

In [ ]:
xFreq2, FFT_data2 = PositiveFFT(y_new, 2.5)

In [ ]:
plt.plot(xFreq2, abs(FFT_data2))

In [ ]:
peaks2, prop2 = find_peaks(abs(FFT_data2), height=0.2)

In [ ]:
peaks2

In [ ]:
xFreq2[7]

In [ ]:
plt.plot(y_gt)

In [ ]:
plt.plot(y_new)

In [ ]:
y_new.shape

In [ ]:
result_gt = adfuller(y_gt)

In [ ]:
result = adfuller(y)

In [ ]:
result = adfuller(y)
print('ADF Statistic: %f' % result[0])
print('p-value: %f' % result[1])
print('Critical Values:')
for key, value in result[4].items():
    print('\t%s: %.3f' % (key, value))

In [ ]:
result_gt = adfuller(y_new)
print('ADF Statistic: %f' % result_gt[0])
print('p-value: %f' % result_gt[1])
print('Critical Values:')
for key, value in result_gt[4].items():
    print('\t%s: %.3f' % (key, value))

In [ ]:
of_video, y_val = tracking_movement(ep_153, m_153, bbx_size=50)
time = np.array(list(range(len(y)))) / 2.5
y = processing(y_val)
plt.plot(time, y)

In [ ]:
epoch = 153
load = os.path.join(path, str(epoch)+'.npy')
os.path.exists(load)

y_gt = np.load(load)
time_gt = np.array(list(range(len(y_gt)))) / 200
time_gt

plt.plot(time, y, time_gt, y_gt)

## Stat

In [ ]:
# epoch 153
plt.plot(time, y, time_gt, y_gt)

In [ ]:
from scipy.signal import find_peaks

In [ ]:
peaks, prop = find_peaks(y, height=0.1)

In [ ]:
peaks

In [ ]:
intv = []
for i in range(len(peaks)-1):
    intv.append(peaks[i+1] - peaks[i])

In [ ]:
intv

In [ ]:
print(f'Index of each peaks : {peaks}')
print(f'Height of eack peaks : {prop["peak_heights"]}')

In [ ]:
plt.plot(y)
plt.plot(peaks, y[peaks], "x")

In [ ]:
np.array(intv) / 2.5

- https://turtle-dennis.tistory.com/21

In [ ]:
peaks, prop = find_peaks(y_gt, height=0.1)
intv = []
for i in range(len(peaks)-1):
    intv.append(peaks[i+1] - peaks[i])
print(intv)

In [ ]:
peaks, prop = find_peaks(y_gt, height=0.1)
intv = []
for i in range(len(peaks)-1):
    intv.append(peaks[i+1] - peaks[i])
print(intv)

plt.plot(y_gt)
plt.plot(peaks, y_gt[peaks], "x")

In [ ]:
np.array(intv) / 200